In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

df_m = pd.read_csv('/Users/kultumlhabaik/Documents/data-mining-final-project/data/raw/dataset_ M.csv')

print(df_m.shape)
df_m.head()

(114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [4]:
df_m.info()

<class 'pandas.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  str    
 2   artists           113999 non-null  str    
 3   album_name        113999 non-null  str    
 4   track_name        113999 non-null  str    
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          114000 non-nu

In [5]:
# check for missing values
missing = df_m.isnull().sum()
print(missing[missing > 0])

artists       1
album_name    1
track_name    1
dtype: int64


In [ ]:
#gets rid of rows with missing values
df_m = df_m.dropna(subset=['artists', 'album_name', 'track_name'])
print('Total Rows :', len(df_m))
#gets rid of unnamed column
df_m = df_m.drop(columns=['Unnamed: 0'])
print('Total Columns:', df_m.shape[1])

Total Rows : 113999
Total Columns: 20


In [6]:
# how many genres are there and how many tracks per genre
print('Total genres:', df_m['track_genre'].nunique())
print('\nTracks per genre:')
print(df_m['track_genre'].value_counts())

Total genres: 114

Tracks per genre:
track_genre
acoustic       1000
afrobeat       1000
alt-rock       1000
alternative    1000
ambient        1000
               ... 
techno         1000
trance         1000
trip-hop       1000
turkish        1000
world-music    1000
Name: count, Length: 114, dtype: int64


In [9]:
# see how useful a join will be based off the amount of track_ids from dataset_M that exist in the main dataset


ja_ids = set(pd.read_csv('/Users/kultumlhabaik/Documents/data-mining-final-project/data/raw/spotify_songs_JA.csv')['track_id'])
m_ids = set(df_m['track_id'])

overlap = ja_ids.intersection(m_ids)
print('Tracks in JA dataset:       ', len(ja_ids))
print('Tracks in M dataset:        ', len(m_ids))
print('Tracks in both (overlap):   ', len(overlap))
print('Overlap percentage:         ', round(len(overlap) / len(ja_ids) * 100, 1), '%')

Tracks in JA dataset:        28356
Tracks in M dataset:         89740
Tracks in both (overlap):    1964
Overlap percentage:          6.9 %


In [10]:
# instead of matching by track, match by artist name
# this shows how many artists appear in both datasets

ja_artists = set(pd.read_csv('/Users/kultumlhabaik/Documents/data-mining-final-project/data/raw/spotify_songs_JA.csv')['track_artist'].dropna())
m_artists = set(df_m['artists'].dropna())

overlap_artists = ja_artists.intersection(m_artists)
print('Artists in JA dataset:      ', len(ja_artists))
print('Artists in M dataset:       ', len(m_artists))
print('Artists in both:            ', len(overlap_artists))
print('Overlap percentage:         ', round(len(overlap_artists) / len(ja_artists) * 100, 1), '%')

Artists in JA dataset:       10692
Artists in M dataset:        31437
Artists in both:             2229
Overlap percentage:          20.8 %


## Dataset Investigation Summary

### Track-Level Join (track_id)
Overlap: 1,964 out of 28,356 tracks (6.9%)
This is too low to be useful. Only 7% of tracks would get
genre labels and the other 93% would be left blank.

### Artist-Level Join (artist name)
Overlap: 2,229 out of 10,692 artists (20.8%)
Better but still not good enough. 79% of artists would
have no genre information at all.

### Why Genre Labels Are Unreliable Anyway
The JA dataset only has 6 broad playlist-based genre labels
which are too coarse for meaningful analysis. The dataset_M
genres are better but without sufficient overlap they cannot
be applied to most of our artists.

### Decision
We are dropping genre as a feature entirely and replacing it
with audio features from spotify_songs_JA.csv which are
complete, precise, and more analytically interesting.

Instead of asking "what genre is this artist" we will ask:
- Is a declining artist's music getting less energetic over time?
- Do artists with higher valence sustain longer careers?
- Does danceability predict whether an artist stays popular?

These questions are more compelling and fully supported by
our existing data with no missing values or join issues.

### Final Decision: Use Only spotify_songs_JA.csv
All artist-year profiles will be built from this single
dataset using popularity, audio features, track count,
and release behavior. dataset_M.csv will not be used.